In [ ]:
from google.colab import drive
drive.mount("/content/drive")


# Semantic-texture soft-detector asset preparation

The first cell mounts Drive. Run all once in a fresh GPU runtime. This stable notebook clones the latest GitHub main branch into fresh `/content`, records the observed checkout revision, copies the sole Drive checkpoint input into `/content`, invokes only the repository bootstrap, and exports one create-only diagnostic-only asset bundle.


In [ ]:
from hashlib import sha256
import json
import os
from pathlib import Path
import shutil
import stat
import subprocess
import sys
from uuid import uuid4
import zipfile

from google.colab import userdata

sys.tracebacklimit = 0
PROJECT_REPOSITORY_URL = "https://github.com/RICHAAARC/CEG-WM.git"
PROJECT_BRANCH = "main"
CHECKPOINT_RELATIVE_PATH = Path("MyDrive/CEG-WM/models/inspyrenet/ckpt_base.pth")
CHECKSUMS_FILENAME = "SHA256SUMS"
BUNDLE_FILENAME = "semantic_texture_soft_detector_asset_bundle.json"
RESULT_FILENAME = "semantic_texture_soft_detector_asset_result.json"
RECEIPT_FILENAME = "semantic_texture_soft_detector_asset_receipt.json"
TRANSPORT_RESULT_FILENAME = "semantic_texture_soft_detector_asset_transport_result.json"
TRANSPORT_RECEIPT_FILENAME = "semantic_texture_soft_detector_asset_transport_receipt.json"

def _sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _canonical_digest(value: object) -> str:
    return sha256(json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode("utf-8")).hexdigest()

def _copy_checkpoint_create_only(source: Path, destination: Path) -> None:
    source_stat = source.lstat()
    if not stat.S_ISREG(source_stat.st_mode) or source.name != "ckpt_base.pth" or destination.exists():
        raise RuntimeError("checkpoint copy boundary failed")
    destination.parent.mkdir(parents=True, exist_ok=False)
    with source.open("rb") as input_handle, destination.open("xb") as output_handle:
        shutil.copyfileobj(input_handle, output_handle)

def _persist_preclone_transport(root: Path, run_id: str, blocked_class: str) -> None:
    if root.exists() or blocked_class not in {"environment_blocked", "resource_blocked", "implementation_blocked", "integrity_blocked"}:
        raise RuntimeError("pre-clone transport boundary failed")
    root.mkdir(parents=True)
    result = {"blocked_class": blocked_class, "candidate_promoted": False, "diagnostic_only": True, "formal_tau_created": False, "profile_id": "semantic_texture_soft_detector_asset_preparation_transport", "run_id": run_id, "science_started": False, "scientific_claims_supported": False, "scientific_unit_count": 0, "status": "blocked", "transport_kind": "notebook_preclone_failure"}
    result_blob = (json.dumps(result, allow_nan=False, indent=2, sort_keys=True) + "\n").encode("utf-8")
    archive_name = f"semantic_texture_soft_detector_assets_{run_id}.zip"
    with (root / TRANSPORT_RESULT_FILENAME).open("xb") as handle:
        handle.write(result_blob)
    with zipfile.ZipFile(root / archive_name, mode="x", compression=zipfile.ZIP_STORED) as archive:
        archive.writestr(TRANSPORT_RESULT_FILENAME, result_blob)
    receipt = {"archive_filename": archive_name, "archive_sha256": _sha256_file(root / archive_name), "blocked_class": blocked_class, "result_filename": TRANSPORT_RESULT_FILENAME, "result_sha256": sha256(result_blob).hexdigest(), "run_id": run_id, "status": "blocked"}
    receipt_blob = (json.dumps(receipt, allow_nan=False, indent=2, sort_keys=True) + "\n").encode("utf-8")
    with (root / TRANSPORT_RECEIPT_FILENAME).open("xb") as handle:
        handle.write(receipt_blob)
    with (root / CHECKSUMS_FILENAME).open("xb") as handle:
        handle.write(f"{_sha256_file(root / TRANSPORT_RESULT_FILENAME)}  {TRANSPORT_RESULT_FILENAME}\n{_sha256_file(root / archive_name)}  {archive_name}\n{_sha256_file(root / TRANSPORT_RECEIPT_FILENAME)}  {TRANSPORT_RECEIPT_FILENAME}\n".encode("ascii"))

def _validate_asset_delivery(root: Path, run_id: str) -> tuple[tuple[str, ...], str, dict[str, tuple[int, str]]]:
    bundle_path, result_path, receipt_path = root / BUNDLE_FILENAME, root / RESULT_FILENAME, root / RECEIPT_FILENAME
    if not bundle_path.is_file() or not result_path.is_file() or not receipt_path.is_file():
        raise RuntimeError("asset delivery is incomplete")
    bundle = json.loads(bundle_path.read_text(encoding="utf-8"))
    bundle_digest = bundle.pop("bundle_digest", None)
    if not isinstance(bundle_digest, str) or _canonical_digest(bundle) != bundle_digest:
        raise RuntimeError("asset bundle identity drifted")
    archive_name = f"semantic_texture_soft_detector_assets_{bundle_digest}.zip"
    names = (BUNDLE_FILENAME, RESULT_FILENAME, archive_name, RECEIPT_FILENAME, CHECKSUMS_FILENAME)
    if {path.name for path in root.iterdir() if path.is_file()} != set(names):
        raise RuntimeError("asset delivery roster drifted")
    result_blob = result_path.read_bytes()
    expected = {"asset_bundle_digest": bundle_digest, "candidate_promoted": False, "diagnostic_only": True, "formal_tau_created": False, "observed_repository_revision": observed_repository_revision, "profile_id": "semantic_texture_soft_detector_asset_preparation", "run_id": run_id, "science_started": False, "scientific_claims_supported": False, "scientific_unit_count": 0, "status": "passed"}
    if json.loads(result_blob) != expected:
        raise RuntimeError("asset result boundary drifted")
    archive_path = root / archive_name
    with zipfile.ZipFile(archive_path) as archive:
        if archive.namelist() != [RESULT_FILENAME] or archive.read(RESULT_FILENAME) != result_blob:
            raise RuntimeError("asset result-only archive boundary drifted")
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    if receipt.get("asset_bundle_digest") != bundle_digest or receipt.get("bundle_sha256") != _sha256_file(bundle_path) or receipt.get("result_sha256") != _sha256_file(result_path) or receipt.get("zip_sha256") != _sha256_file(archive_path):
        raise RuntimeError("asset receipt binding drifted")
    lines = (root / CHECKSUMS_FILENAME).read_text(encoding="ascii").splitlines()
    if [line.split("  ", 1)[1] for line in lines] != list(names[:4]):
        raise RuntimeError("asset checksum roster drifted")
    for line in lines:
        digest, name = line.split("  ", 1)
        if _sha256_file(root / name) != digest:
            raise RuntimeError("asset checksum binding drifted")
    return names, bundle_digest, {name: ((root / name).stat().st_size, _sha256_file(root / name)) for name in names}

def _validate_transport_delivery(root: Path, run_id: str) -> tuple[tuple[str, ...], dict[str, tuple[int, str]]]:
    archive_name = f"semantic_texture_soft_detector_assets_{run_id}.zip"
    names = (TRANSPORT_RESULT_FILENAME, archive_name, TRANSPORT_RECEIPT_FILENAME, CHECKSUMS_FILENAME)
    if not root.is_dir() or {path.name for path in root.iterdir() if path.is_file()} != set(names):
        raise RuntimeError("transport delivery roster drifted")
    result_blob = (root / TRANSPORT_RESULT_FILENAME).read_bytes()
    result = json.loads(result_blob)
    if result.get("status") != "blocked" or result.get("science_started") is not False or result.get("scientific_unit_count") != 0 or result.get("candidate_promoted") is not False:
        raise RuntimeError("transport zero-science boundary drifted")
    with zipfile.ZipFile(root / archive_name) as archive:
        if archive.namelist() != [TRANSPORT_RESULT_FILENAME] or archive.read(TRANSPORT_RESULT_FILENAME) != result_blob:
            raise RuntimeError("transport result-only archive drifted")
    lines = (root / CHECKSUMS_FILENAME).read_text(encoding="ascii").splitlines()
    if [line.split("  ", 1)[1] for line in lines] != list(names[:3]):
        raise RuntimeError("transport checksum roster drifted")
    for line in lines:
        digest, name = line.split("  ", 1)
        if _sha256_file(root / name) != digest:
            raise RuntimeError("transport checksum binding drifted")
    return names, {name: ((root / name).stat().st_size, _sha256_file(root / name)) for name in names}

def _validate_asset_blocked_delivery(root: Path, run_id: str) -> tuple[tuple[str, ...], dict[str, tuple[int, str]]]:
    archive_name = f"semantic_texture_soft_detector_assets_{run_id}.zip"
    names = (RESULT_FILENAME, archive_name, RECEIPT_FILENAME, CHECKSUMS_FILENAME)
    if not root.is_dir() or {path.name for path in root.iterdir() if path.is_file()} != set(names):
        raise RuntimeError("blocked asset delivery roster drifted")
    result_blob = (root / RESULT_FILENAME).read_bytes()
    result = json.loads(result_blob)
    if result.get("status") != "blocked" or result.get("science_started") is not False or result.get("scientific_unit_count") != 0 or result.get("candidate_promoted") is not False:
        raise RuntimeError("blocked asset zero-science boundary drifted")
    with zipfile.ZipFile(root / archive_name) as archive:
        if archive.namelist() != [RESULT_FILENAME] or archive.read(RESULT_FILENAME) != result_blob:
            raise RuntimeError("blocked asset result-only archive drifted")
    lines = (root / CHECKSUMS_FILENAME).read_text(encoding="ascii").splitlines()
    if [line.split("  ", 1)[1] for line in lines] != list(names[:3]):
        raise RuntimeError("blocked asset checksum roster drifted")
    for line in lines:
        digest, name = line.split("  ", 1)
        if _sha256_file(root / name) != digest:
            raise RuntimeError("blocked asset checksum binding drifted")
    return names, {name: ((root / name).stat().st_size, _sha256_file(root / name)) for name in names}


In [ ]:
mount_run_id = "semantic-texture-soft-detector-assets-mount-" + uuid4().hex
content_root = Path("/content")
local_parent = content_root / "ceg-wm-semantic-texture-soft-detector-assets"
mount_root = local_parent / mount_run_id
if mount_root.exists():
    raise RuntimeError("fresh local root is required")
mount_root.mkdir(parents=True)
exports_parent = Path("/content/drive/MyDrive/CEG-WM/semantic_texture_soft_detector_assets")
exports_parent.mkdir(parents=True, exist_ok=True)
run_id = "semantic-texture-soft-detector-assets-" + uuid4().hex
local_root = local_parent / run_id
local_root.mkdir()
checkout_root = local_root / "repository"
execution_root = local_root / "execution"
checkpoint_path = local_root / "assets/inspyrenet/ckpt_base.pth"
try:
    subprocess.run(["git", "clone", "--branch", PROJECT_BRANCH, "--single-branch", PROJECT_REPOSITORY_URL, str(checkout_root)], check=True, capture_output=True)
    observed_repository_revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=checkout_root, check=True, capture_output=True, text=True).stdout.strip()
    _copy_checkpoint_create_only(Path("/content/drive") / CHECKPOINT_RELATIVE_PATH, checkpoint_path)
    hf_token = userdata.get("HF_TOKEN")
    root_key = userdata.get("CEG_WM_ROOT_KEY")
    if not isinstance(hf_token, str) or not hf_token or not isinstance(root_key, str) or not root_key:
        raise RuntimeError("required Colab Secrets are unavailable")
except Exception as error:
    blocked_class = "resource_blocked" if isinstance(error, (MemoryError, OSError)) else "environment_blocked" if isinstance(error, (subprocess.SubprocessError, RuntimeError)) else "implementation_blocked"
    _persist_preclone_transport(exports_parent / run_id, run_id, blocked_class)
    raise RuntimeError("asset preparation bootstrap blocked") from None


In [ ]:
environment = os.environ.copy()
environment.update({"HF_TOKEN": hf_token, "CEG_WM_ROOT_KEY": root_key, "PYTHONDONTWRITEBYTECODE": "1"})
bootstrap_command = [sys.executable, str(checkout_root / "scripts/experiment_execution/semantic_texture_operational_preflight_bootstrap.py"), "--repository-root", str(checkout_root), "--checkpoint", str(checkpoint_path), "--execution-root", str(execution_root), "--entrypoint-path", "scripts/experiment_execution/semantic_texture_soft_detector_asset_preparation_entrypoint.py", "--entrypoint-args", "--execute", "--observed-repository-revision", observed_repository_revision, "--run-id", run_id, "--output-root", str(local_root / "asset-delivery")]
try:
    completed = subprocess.run(bootstrap_command, check=False, capture_output=True, env=environment)
finally:
    del environment, hf_token, root_key
asset_parent = local_root / "asset-delivery"
asset_roots = tuple(path for path in asset_parent.iterdir() if path.is_dir()) if asset_parent.is_dir() else ()
if completed.returncode == 0:
    if len(asset_roots) != 1:
        raise RuntimeError("asset bundle root is ambiguous")
    artifact_root = asset_roots[0]
    artifact_names, bundle_digest, identities = _validate_asset_delivery(artifact_root, run_id)
    drive_export_root = exports_parent / bundle_digest
elif len(asset_roots) == 1:
    artifact_root = asset_roots[0]
    artifact_names, identities = _validate_asset_blocked_delivery(artifact_root, run_id)
    drive_export_root = exports_parent / run_id
else:
    artifact_root = execution_root.with_name(execution_root.name + ".transport")
    artifact_names, identities = _validate_transport_delivery(artifact_root, run_id)
    drive_export_root = exports_parent / run_id
if drive_export_root.exists():
    raise RuntimeError("asset delivery collision blocked")
drive_export_root.mkdir()
for name in artifact_names[:3] if completed.returncode != 0 else artifact_names[:4]:
    source = artifact_root / name
    destination = drive_export_root / name
    with source.open("rb") as input_handle, destination.open("xb") as output_handle:
        shutil.copyfileobj(input_handle, output_handle)
    if (destination.stat().st_size, _sha256_file(destination)) != identities[name]:
        raise RuntimeError("Drive asset identity drifted")
completion_blob = (artifact_root / CHECKSUMS_FILENAME).read_bytes()
pending = drive_export_root / ".completion.pending"
with pending.open("xb") as output_handle:
    output_handle.write(completion_blob)
    output_handle.flush()
    os.fsync(output_handle.fileno())
os.replace(pending, drive_export_root / CHECKSUMS_FILENAME)
if (drive_export_root / CHECKSUMS_FILENAME).read_bytes() != completion_blob:
    raise RuntimeError("asset completion marker changed during Drive delivery")
if completed.returncode != 0:
    raise RuntimeError("asset preparation blocked after complete Drive delivery") from None
